# Dev32: Apply 028 SVM Model to 057 Transect

**Goal**: Use the trained SVM from dev30 (trained on 028) and apply it to 057 transect.

**Workflow**:
1. Load saved SVM model from dev30
2. Load and preprocess 057 transect (same preprocessing as dev30)
3. Classify entire segment
4. Apply filtering (remove small clusters)
5. Visualize with plot_georef (like dev20)

## Setup and Import

In [ ]:
import importlib
import sys
from pathlib import Path
import pickle


# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent  # Up to mjosa_code root
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code (NEW structure)
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)
from utils.uhi.georef import *

## Step 1: Load Trained SVM Model from dev30

**Note**: First run the save cell in dev30 to create the model file!

In [ ]:
# Load the saved SVM model and preprocessing info
model_file = "./saved_data/svm_model_028_multiclass.pkl"

if not os.path.exists(model_file):
    print("❌ ERROR: Model file not found!")
    print(f"   Looking for: {model_file}")
    print("\n📝 TO CREATE THE MODEL FILE:")
    print("   1. Open dev30 notebook")
    print("   2. Run all cells up to 'Train SVM'")
    print("   3. Run the '💾 SAVE SVM MODEL' cell")
    print("   4. Then come back here!")
    raise FileNotFoundError(f"Model file not found: {model_file}")
else:
    with open(model_file, "rb") as f:
        saved_model_data = pickle.load(f)

    # Extract model and parameters
    trained_model = saved_model_data["model"]
    class_names = saved_model_data["class_names"]
    label_encoder = saved_model_data["label_encoder"]
    preprocessing_params = saved_model_data["preprocessing"]
    best_params = saved_model_data["best_params"]

    print("✅ SVM model loaded successfully!")
    print(f"\n📊 Model Info:")
    print(f"   Classes: {class_names}")
    print(f"   Best C: {best_params['C']}")
    print(f"   Best gamma: {best_params['gamma']}")
    print(f"\n🔬 Preprocessing (will apply to 057):")
    print(
        f"   Smoothing: {preprocessing_params['smoothing_method']} (σ={preprocessing_params['smoothing_sigma']})"
    )
    print(f"   Wavelength range: {preprocessing_params['wavelength_range']} nm")
    print(f"   Normalization: {preprocessing_params['normalization']}")

## Step 2: Load 057 Transect (File 5)

In [ ]:
# Load transect
transect = load_transect(config.TRANSECT_057_OUTPUT)
transect.list_files()

# Select file
cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()
print("✅ Illumination correction applied")

## Step 3: Apply Same Preprocessing as dev30

In [ ]:
# Apply spectral smoothing (same as dev30)
cube.apply_spectral_smoothing(
    method=preprocessing_params["smoothing_method"],
    gaussian_sigma=preprocessing_params["smoothing_sigma"],
)
print(
    f"✅ Spectral smoothing applied: {preprocessing_params['smoothing_method']} (σ={preprocessing_params['smoothing_sigma']})"
)

In [ ]:
# Crop wavelengths (same as dev30)
wl_min, wl_max = preprocessing_params["wavelength_range"]
cube.apply_wavelength_filter(wavelength_range=(wl_min, wl_max))

print(f"✅ Wavelength cropping complete")
print(f"   Cropped shape: {cube.data_corrected.shape}")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")

In [ ]:
# NO L2 normalization (like dev30)
print("✅ No normalization applied (matching dev30 preprocessing)")

## Step 4: Classify Entire Segment (using dev30's model)

**Note**: We're using the trained model from 028, applying it to 057 data.

## Step 3.5: Set the Loaded Model on the Cube

The `classify_segment()` method uses the model stored in the cube object. We need to set it manually.

In [ ]:
# Set the loaded model on the cube object
# (classify_segment() expects these attributes to exist)
cube.svm_model = trained_model
cube.svm_label_encoder = label_encoder
cube.svm_class_names = class_names

print("✅ Model set on cube object")
print(f"   cube.svm_model: {type(cube.svm_model)}")
print(f"   cube.svm_label_encoder: {type(cube.svm_label_encoder)}")
print(f"   cube.svm_class_names: {cube.svm_class_names}")

In [ ]:
# Get track range for file 5 (from config or manually)
track_start = config.UHI_TRACK_RANGE[0] if hasattr(config, "UHI_TRACK_RANGE") else 0
track_end = (
    config.UHI_TRACK_RANGE[1]
    if hasattr(config, "UHI_TRACK_RANGE")
    else cube.data_corrected.shape[0]
)

print(f"🔍 Track range for classification: [{track_start}, {track_end}]")
print(f"   Total tracks: {track_end - track_start}")

In [ ]:
# Classify using the loaded model from dev30
print("\n🎯 Classifying 057 transect using 028-trained model...")
print("=" * 60)

classification_results = cube.classify_segment(
    segment_start=track_start,
    segment_end=track_end - 1,
    use_corrected=True,
    confidence_threshold=0,  # Reject low-confidence pixels → classified_unknown
    quiet=False,
)

print(f"\n✅ Classification complete!")
print(f"   Classes: {classification_results['class_names']}")
print(f"\n📊 Pixel counts:")
for class_name in classification_results["class_names"]:
    count = np.sum(classification_results["classification_map"] == class_name)
    percentage = 100.0 * count / classification_results["classification_map"].size
    print(f"   {class_name}: {count} pixels ({percentage:.1f}%)")

## Step 5: Visualize Classification Results

Plot the raw classification (before filtering)

In [ ]:
%matplotlib inline

# Convert classification map to ROI format for plotting
print("📊 Plotting classification map...")

classification_rois = {}
for class_name in classification_results['class_names']:
    mask = classification_results['classification_map'] == class_name
    rows, cols = np.where(mask)
    # Convert to global coordinates (add track_start offset)
    pixels = [(col, row + track_start) for row, col in zip(rows, cols)]
    if len(pixels) > 0:
        classification_rois[class_name] = pixels

print(f"   ROIs created: {list(classification_rois.keys())}")
print(f"   Pixel counts:")
for class_name in sorted(classification_rois.keys()):
    print(f"      {class_name}: {len(classification_rois[class_name])} pixels")

# Plot with plot_georef (like dev20)
cube.plot_georef(
    use_corrected=True,
    coordinate_system="NED",
    track_start=track_start,
    track_end=track_end,
    figsize=(40, 10),
    roi_collection=classification_rois,
    roi_overlay_mode="solid",

    roi_marker_size=1,
    roi_legend_loc="outside",
    roi_legend_markersize=50,
)

## Step 6: Apply Filtering (like dev20)

Remove small clusters of dark_bomb, dark_pit, and halo (similar to dev20's filtering)

In [ ]:
# Apply filtering to remove small clusters
print("\n🧹 FILTERING: Removing small clusters...")
print("=" * 60)

# Define which classes to filter (keep sediment and rust as is)
classes_to_filter = ["dark_bomb", "dark_pit", "halo"]

filtering_results = cube.filter_classification(
    classification_map=classification_results["classification_map"],
    class_names=classification_results["class_names"],
    filter_bombs=True,  # Filter dark_bomb
    filter_dark=True,  # Filter dark_pit and halo
    min_area_px=10,  # Minimum cluster size (pixels)
    quiet=False,
)

print(f"\n✅ Filtering complete!")
print(f"\n📊 Pixel counts after filtering:")
# Use 'all_classes' which contains all unique classes in filtered map
for class_name in filtering_results["all_classes"]:
    count = np.sum(filtering_results["filtered_map"] == class_name)
    percentage = 100.0 * count / filtering_results["filtered_map"].size
    print(f"   {class_name}: {count} pixels ({percentage:.1f}%)")

In [ ]:
# DEBUG: Check what classes exist before filtering
print("🔍 DEBUG: Classes in classification_results:")
print(f"   class_names: {classification_results['class_names']}")
print(f"\n   Unique classes in map:")
for cn in np.unique(classification_results["classification_map"]):
    count = np.sum(classification_results["classification_map"] == cn)
    print(f"      {cn}: {count} pixels")

In [ ]:
# Reload georef module to get the bug fix
import importlib

importlib.reload(georef)
print("✅ Reloaded georef module with bug fix")

## Step 7: Visualize Filtered Results

In [ ]:
# Convert filtered map to ROI format
print("📊 Plotting filtered classification map...")

filtered_rois = {}
# Use 'all_classes' which contains all unique classes in filtered map
for class_name in filtering_results["all_classes"]:
    mask = filtering_results["filtered_map"] == class_name
    rows, cols = np.where(mask)
    pixels = [(col, row + track_start) for row, col in zip(rows, cols)]
    if len(pixels) > 0:
        filtered_rois[class_name] = pixels

print(f"   Filtered ROIs: {list(filtered_rois.keys())}")
print(f"   Pixel counts:")
for class_name in sorted(filtered_rois.keys()):
    print(f"      {class_name}: {len(filtered_rois[class_name])} pixels")

# Plot filtered results
cube.plot_georef(
    use_corrected=True,
    coordinate_system="NED",
    track_start=track_start,
    track_end=track_end,
    figsize=(40, 10),
    roi_collection=filtered_rois,
    roi_overlay_mode="solid",
    roi_solid_pixel_size=1,
    roi_legend_loc="outside",
    roi_legend_markersize=50,
)

## Summary

**What we did:**
1. ✅ Loaded SVM model trained on 028 transect (dev30)
2. ✅ Applied same preprocessing to 057 transect
3. ✅ Classified 057 using 028's model (cross-transect transfer!)
4. ✅ Applied filtering to remove small clusters
5. ✅ Visualized results with plot_georef

**Key insight**: Testing if the 028-trained model generalizes to 057!

---

## 📊 COMPREHENSIVE CLASSIFICATION ANALYSIS

Generate detailed metrics, visualizations, and statistics for cross-transect transfer (028 model → 057 data).

In [ ]:
# Import the comprehensive analysis function
from utils.uhi.classification_analysis import analyze_classification_results

print("=" * 80)
print("📊 CROSS-TRANSECT TRANSFER ANALYSIS: 028 MODEL → 057 DATA")
print("=" * 80)
print(f"Model: SVM trained on transect 028 (5-class)")
print(f"Application: Applied to transect 057")
print(f"Purpose: Test model generalization across different seafloor sites")
print("=" * 80)

# Gather classification and filtering results
cls_map = (
    classification_results.get("classification_map")
    if "classification_results" in globals()
    else None
)
filt_map = (
    filtering_results.get("filtered_map") if "filtering_results" in globals() else None
)
class_names_var = (
    classification_results.get("class_names")
    if "classification_results" in globals()
    else None
)

if cls_map is None:
    print("❌ ERROR: classification_results not found!")
    print("   Please run the classification cells first.")
else:
    # Run comprehensive analysis
    metrics_028_to_057 = analyze_classification_results(
        classification_map=cls_map,
        filtered_map=filt_map,
        class_names=class_names_var,
        track_start=track_start if "track_start" in globals() else 0,
        figsize=(14, 6),
        show_plots=True,
        save_csv="./saved_data/classification_summary_028_to_057.csv",
    )

    print("\n" + "=" * 80)
    print("📝 CROSS-TRANSFER NARRATIVE SUMMARY")
    print("=" * 80)
    print(f"The 028 model (trained on 5 classes) was applied to 057 transect.")
    print(f"Total pixels classified: {metrics_028_to_057['total_pixels']:,}")
    print(f"\nClass distribution in 057 when using 028 model:")

    if metrics_028_to_057["counts_after"] is not None:
        for cn in class_names_var:
            before = metrics_028_to_057["counts_before"].get(cn, 0)
            after = metrics_028_to_057["counts_after"].get(cn, 0)
            pct_after = 100.0 * after / metrics_028_to_057["total_pixels"]
            removed = before - after
            print(f"  • {cn}:")
            print(f"      Before filtering: {before:,} pixels")
            print(f"      After filtering: {after:,} pixels ({pct_after:.2f}%)")
            if removed > 0:
                print(f"      Removed: {removed:,} pixels (small clusters)")
    else:
        for cn in class_names_var:
            count = metrics_028_to_057["counts_before"].get(cn, 0)
            pct = 100.0 * count / metrics_028_to_057["total_pixels"]
            print(f"  • {cn}: {count:,} pixels ({pct:.2f}%)")

    print("\n🔍 KEY INSIGHT:")
    print("   Compare these results with notebook 6 (057 model on 057 data)")
    print("   to assess how well the 028 model generalizes to a different site.")
    print("=" * 80)